# Imported

In [17]:
import numpy as np
import csv
import random
import os
import torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    AutoConfig,
    DataCollatorForTokenClassification,
    AutoModelForTokenClassification
)
import json

# Fine tune Instruction

## Generate dataset

In [2]:
import random
import pandas as pd

random.seed(42)

prefixes = ["", "tolong ", "coba ", "robot ", "hei ", "bot ", "minta tolong "]
suffixes = ["", " ya", " dong", " sekarang", " sebentar", " di sana"]

# NAVIGATE_TO_OBJECT
objects = [
    "meja", "kursi", "pintu", "laptop", "botol", "lemari", "kulkas",
    "televisi", "papan tulis", "kasur", "sofa", "dispenser", "tempat sampah",
    "gelas", "sepatu", "tas", "buku", "proyektor", "jendela", "kamarmandi"
]
obj_positions = ["", " di depan", " di sana", " sebelah kiri", " sebelah kanan", " dekat dinding"]

# MOVE_RELATIVE
rel_verbs = ["jalan", "maju", "mundur", "geser", "bergerak", "pindah"]
rel_directions = ["ke depan", "ke belakang", "ke kiri", "ke kanan", "lurus"]
distances = ["1", "2", "3", "0.5", "setengah", "sedikit", "beberapa"]
units = ["meter", "m", "cm", "langkah"]

# untuk ROTATE
rot_verbs = ["putar", "berputar", "rotasi", "belok", "mengarahkan badan", "tengok", "hadap"]
rot_directions = ["ke kiri", "ke kanan", "ke belakang", "kiri", "kanan", "belakang"]
degrees = ["30", "45", "60", "90", "180", "360"]

def generate_navigate_samples(count=100):
    samples = set()
    actions = ["jalan ke", "pergi ke", "menuju ke", "samperin", "dekati", "bergerak ke", "cari"]

    while len(samples) < count:
        p = random.choice(prefixes)
        a = random.choice(actions)
        o = random.choice(objects)
        pos = random.choice(obj_positions)
        s = random.choice(suffixes)

        sentence = f"{p}{a} {o}{pos}{s}".strip().lower()

        sentence = " ".join(sentence.split())
        samples.add(sentence)

    return list(samples)

def generate_move_relative_samples(count=100):
    samples = set()

    while len(samples) < count:
        p = random.choice(prefixes)
        v = random.choice(rel_verbs)
        d = random.choice(rel_directions)
        dist = random.choice(distances)
        u = random.choice(units)
        s = random.choice(suffixes)

        pattern = random.choice([1, 2, 3])
        if pattern == 1:
            sentence = f"{p}{v} {d} {dist} {u}{s}"
        elif pattern == 2:
            sentence = f"{p}{v} {d}{s}"
        else:
            sentence = f"{p}{v} {dist} {u} {d}{s}"

        sentence = " ".join(sentence.strip().lower().split())
        samples.add(sentence)

    return list(samples)

def generate_rotate_samples(count=100):
    samples = set()

    while len(samples) < count:
        p = random.choice(prefixes)
        v = random.choice(rot_verbs)
        d = random.choice(rot_directions)
        deg = random.choice(degrees)
        s = random.choice(suffixes)

        pattern = random.choice([1, 2, 3])
        if pattern == 1:
            sentence = f"{p}{v} {d} {deg} derajat{s}"
        elif pattern == 2:
            sentence = f"{p}{v} {d}{s}"
        else:
            sentence = f"{p}{v} {deg} derajat {d}{s}"

        sentence = " ".join(sentence.strip().lower().split())
        samples.add(sentence)

    return list(samples)

print("Menghasilkan dataset...")
nav_data = generate_navigate_samples(100)
move_data = generate_move_relative_samples(100)
rot_data = generate_rotate_samples(100)

all_texts = nav_data + move_data + rot_data
all_labels = [0] * 100 + [1] * 100 + [2] * 100

data_pairs = list(zip(all_texts, all_labels))
random.shuffle(data_pairs)

df = pd.DataFrame(data_pairs, columns=["text", "label"])

csv_filename = "dataset_robot_slam.csv"
df.to_csv(csv_filename, index=False)

print(f"Selesai! {len(df)} baris dataset berhasil disimpan di '{csv_filename}'.")
print("\nPratinjau 5 data pertama:")
print(df.head())

Menghasilkan dataset...
Selesai! 300 baris dataset berhasil disimpan di 'dataset_robot_slam.csv'.

Pratinjau 5 data pertama:
                                               text  label
0  minta tolong mengarahkan badan belakang sekarang      2
1            coba cari kursi dekat dinding sekarang      0
2                    bot geser ke depan 1 m di sana      1
3                minta tolong mundur ke belakang ya      1
4            samperin papan tulis sebelah kiri dong      0


## Preprocess data

In [3]:
data = np.genfromtxt('dataset_robot_slam.csv', delimiter=',', dtype=None, names=True, encoding='utf-8')

data = {
    'text': data['text'],
    'label': data['label']
}

id2label = {0: "NAVIGATE_TO_OBJECT", 1: "MOVE_RELATIVE", 2: "ROTATE"}
label2id = {"NAVIGATE_TO_OBJECT": 0, "MOVE_RELATIVE": 1, "ROTATE": 2}

df = pd.DataFrame(data)
dataset = Dataset.from_pandas(df)

dataset = dataset.train_test_split(test_size=0.2, seed=42)
MODEL_NAME = "indolem/indobert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=32)

tokenized_datasets = dataset.map(preprocess_function, batched=True)

config = AutoConfig.from_pretrained(MODEL_NAME)
config.num_labels = 3
config.id2label = id2label
config.label2id = label2id

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config,
    ignore_mismatched_sizes=True
)

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/234k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Map:   0%|          | 0/60 [00:00<?, ? examples/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  445MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indolem/indobert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Training Process

In [4]:
training_args = TrainingArguments(
    output_dir="./indobert_robot_intent",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_steps=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer,
)

print("Memulai Fine-Tuning IndoBERT...")
trainer.train()
print("Selesai Fine-Tuning IndoBERT!")

output_model_dir = "./saved_indobert_intent_model"
model.save_pretrained(output_model_dir)
tokenizer.save_pretrained(output_model_dir)
print(f"Model berhasil disimpan di folder: {output_model_dir}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Memulai Fine-Tuning IndoBERT...


Epoch,Training Loss,Validation Loss
1,0.991317,0.939064
2,0.781368,0.649384
3,0.453357,0.287972
4,0.526484,0.331905
5,0.094612,0.119397
6,0.075034,0.069944
7,0.012694,0.144949
8,0.307972,0.070123
9,0.002799,0.172632
10,0.005063,0.181936


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Selesai Fine-Tuning IndoBERT!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model berhasil disimpan di folder: ./saved_indobert_intent_model


## Test

In [8]:
from transformers import pipeline

nlp_classifier = pipeline(
    "text-classification",
    model="./saved_indobert_intent_model",
    tokenizer="./saved_indobert_intent_model"
)

word_test = "maju 3 meter"
result = nlp_classifier(word_test)

print(f"Kalimat : '{word_test}'")
print(f"Intent  : {result[0]['label']}")
print(f"Score   : {result[0]['score']:.4f}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Kalimat : 'maju 3 meter'
Intent  : MOVE_RELATIVE
Score   : 0.9986


# Fine tune object

## Generated Dataset

In [9]:
first_word_list = [
    ["tolong", "jalan", "ke"],
    ["pergi", "ke"],
    ["bergerak", "mendekati"],
    ["menuju", "ke"],
    ["cari"],
    ["temukan"],
    ["tolong", "hampiri"],
    ["jalan", "menuju"]
]

object_list = [
    (["meja"], [1]),
    (["kursi"], [1]),
    (["pintu"], [1]),
    (["laptop"], [1]),
    (["botol"], [1]),
    (["gelas"], [1]),
    (["sepatu"], [1]),
    (["buku"], [1]),
    (["tas"], [1]),
    (["sandal"], [1]),
    (["botol", "air"], [1, 2]),
    (["botol", "air", "mineral"], [1, 2, 2]),
    (["meja", "belajar"], [1, 2]),
    (["meja", "makan"], [1, 2]),
    (["kursi", "rodanya"], [1, 2]),
    (["pintu", "utama"], [1, 2]),
    (["laptop", "hitam"], [1, 2]),
    (["tas", "punggung"], [1, 2]),
    (["kotak", "sampah"], [1, 2]),
]

last_word_list = [
    [],
    ["di", "depan"],
    ["di", "sebelah", "kiri"],
    ["di", "sebelah", "kanan"],
    ["yang", "ada", "di", "sana"],
    ["sekarang"],
    ["secepatnya"]
]

def generate_ner_dataset(num_samples=100):
    dataset = []

    for _ in range(num_samples):
        first_word = random.choice(first_word_list)
        object_tokens, object_tags = random.choice(object_list)
        last_word = random.choice(last_word_list)

        tokens = first_word + object_tokens + last_word

        ner_tags = [0] * len(first_word) + object_tags + [0] * len(last_word)

        dataset.append({
            "tokens": tokens,
            "ner_tags": ner_tags
        })

    return dataset

data_ner_generated = generate_ner_dataset(200)

print("Contoh Data Generated:")
print(json.dumps(data_ner_generated[:3], indent=2))

Contoh Data Generated:
[
  {
    "tokens": [
      "pergi",
      "ke",
      "meja",
      "sekarang"
    ],
    "ner_tags": [
      0,
      0,
      1,
      0
    ]
  },
  {
    "tokens": [
      "cari",
      "buku",
      "di",
      "depan"
    ],
    "ner_tags": [
      0,
      1,
      0,
      0
    ]
  },
  {
    "tokens": [
      "bergerak",
      "mendekati",
      "laptop",
      "sekarang"
    ],
    "ner_tags": [
      0,
      0,
      1,
      0
    ]
  }
]


## Preprocess Data

In [10]:
label_list = ["O", "B-OBJ", "I-OBJ"]
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}
raw_dataset = Dataset.from_list(data_ner_generated)

## Alignment & Tokenization

In [11]:
MODEL_NAME = "indolem/indobert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(label[word_idx] if label[word_idx] != 1 else 2)
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = raw_dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

## Model Config & Training

In [18]:
config = AutoConfig.from_pretrained(MODEL_NAME)
config.num_labels = len(id2label)
config.id2label = id2label
config.label2id = label2id

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./indobert_ner_object",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=15,
    weight_decay=0.01,
    logging_steps=1,
    report_to="none"
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
    data_collator=data_collator
)

print("Memulai Fine-Tuning NER...")
trainer.train()
print("Selesai Fine-Tuning NER!")

output_ner_dir = "./saved_indobert_ner_model"
model.save_pretrained(output_ner_dir)
tokenizer.save_pretrained(output_ner_dir)
print(f"Model NER disimpan di: {output_ner_dir}")

[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: indolem/indobert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those param

Memulai Fine-Tuning NER...


Step,Training Loss
1,1.215307
2,0.993311
3,0.746703
4,0.659377
5,0.840242
6,0.973209
7,0.808450
8,0.734005
9,0.748616
10,0.715663


[transformers] Error during conversion: ReadTimeout('The read operation timed out')


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Selesai Fine-Tuning NER!


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model NER disimpan di: ./saved_indobert_ner_model


## Test

In [28]:
from transformers import pipeline

intent_classifier = pipeline(
    "text-classification",
    model="./saved_indobert_intent_model",
    tokenizer="./saved_indobert_intent_model"
)

ner_extractor = pipeline(
    "ner",
    model="./saved_indobert_ner_model",
    tokenizer="./saved_indobert_ner_model",
    aggregation_strategy="simple"
)

word = "Dekati buku itu"

intent_resutl = intent_classifier(word)[0]
ner_result = ner_extractor(word)

target_object = None
for entity in ner_result:
    if entity['entity_group'] == 'OBJ':
        target_object = entity['word']
        break

intent = intent_resutl['label']

target = target_object if intent == "NAVIGATE_TO_OBJECT" else "None"

print(f"Kalimat : {word}")
print(f"Intent  : {intent}")
print(f"Objek   : {target}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Kalimat : Dekati buku itu
Intent  : NAVIGATE_TO_OBJECT
Objek   : buku


## Download result

In [30]:
!zip -r nlp.zip saved_indobert_intent_model
!zip -r ner.zip saved_indobert_ner_model

updating: saved_indobert_intent_model/ (stored 0%)
updating: saved_indobert_intent_model/model.safetensors (deflated 7%)
updating: saved_indobert_intent_model/config.json (deflated 53%)
updating: saved_indobert_intent_model/tokenizer.json (deflated 71%)
updating: saved_indobert_intent_model/tokenizer_config.json (deflated 43%)
  adding: saved_indobert_ner_model/ (stored 0%)
  adding: saved_indobert_ner_model/model.safetensors (deflated 7%)
  adding: saved_indobert_ner_model/config.json (deflated 53%)
  adding: saved_indobert_ner_model/tokenizer.json (deflated 71%)
  adding: saved_indobert_ner_model/tokenizer_config.json (deflated 43%)


In [31]:
from google.colab import files
files.download('nlp.zip')
files.download('ner.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>